<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

# IMU Basics

A Duckiedrone needs to keep itself level with the horizon to stay in the air. The sensor that makes this possible is an __inertial measurement unit (IMU)__.

An IMU is a sensor package whose accelerometer reports a noisy __proper-acceleration measurement__ along its own axes and whose gyroscope reports a noisy __angular-velocity measurement__ about them. It is a core component of every modern smart device, from phones to robots, and on the Duckiedrone it is built into the flight-controller board.

The Duckiedrone puts this data to two different uses:

- __attitude__: the flight controller estimates how the Duckiedrone is tilted and corrects small
  perturbations, which is what keeps it horizontal when hovering.

- __motion__: integrating the measurements over time reconstructs velocity and position,
  although this second use comes with intrinsic limitations.

By the end of the notebook you should be able to answer:

- What does an IMU measure, and what does it not measure?

- How does proper acceleration differ from linear acceleration?

- Why does the `linear_acceleration` field of `sensor_msgs/Imu` not hold linear acceleration on the Duckiedrone?

## Notation

[Table 1](#table-1) lists the mathematical conventions, [Table 2](#table-2) the spaces and structures, and [Table 3](#table-3) the semantic symbols used throughout this learning experience (LX).

<a id="table-1"></a>

Table 1: Mathematical conventions used in equations throughout this LX.

| Notation | Meaning |
| --- | --- |
| $\mathbf{u}$ or $\boldsymbol{\omega}$ | Bold lower-case Latin or Greek letter: a vector |
| $\mathbf{A}$ or $\mathbf{R}$ | Bold upper-case Latin letter: a matrix; $\mathbf{R}_{\mathrm{WB}}$ and $\mathbf{R}_{\mathrm{BS}}$ are rotation matrices |
| $(\cdot)_{\mathrm{S}}$, $(\cdot)_{\mathrm{B}}$, $(\cdot)_{\mathrm{W}}$ | Components expressed in the sensor, body, or world frame |
| $(\cdot)_i$, $(\cdot)_k$, $(\cdot)_{i,k}$ | Cartesian-component, sample-time, and combined component/sample subscripts |
| $\{\cdots\}$ | Set containing the listed elements |
| $\tilde{(\cdot)}$ | Measured quantity |
| $\Delta(\cdot)$ | Difference between two values of the enclosed quantity |
| $\hat{(\cdot)}$ | Estimate |
| $\dot{(\cdot)}$, $\ddot{(\cdot)}$ | First and second time derivatives |
| $(\cdot)(t)$ | Continuous-time value at time $t$ |
| $\int$, $\sum$ | Continuous accumulation and discrete summation |
| $\frac{a}{b}$ | Quotient of $a$ and nonzero $b$ |
| $\sin$, $\cos$, $\arcsin$, $\operatorname{atan2}(y, x)$ | Sine, cosine, inverse sine, and two-argument inverse tangent; the signs of $y$ and $x$ select the quadrant |
| $\det(\mathbf{A})$ | Determinant of the square matrix $\mathbf{A}$ |
| $\begin{bmatrix}\cdots\end{bmatrix}$, $\mathsf{T}$ | Column vector and transpose |
| $\lvert\cdot\rvert$, $\lVert\cdot\rVert$ | Scalar magnitude and Euclidean vector length |
| $\pm$ | Plus or minus |
| $=$, $\approx$, $\in$ | Equality, approximate equality, and set membership |
| $<$, $\leq$, $>$, $\geq$ | Strict and non-strict order relations |

[Table 2](#table-2) lists the spaces and structures used throughout this LX.

<a id="table-2"></a>

Table 2: Spaces and structures used throughout this LX.

| Space or structure | Meaning |
| --- | --- |
| $\mathbb{Z}_{\geq 0}$ | Non-negative integers, $\{0, 1, 2, \ldots\}$ |
| $\mathbb{Z}_{>0}$ | Positive integers, $\{1, 2, \ldots\}$ |
| $\mathbb{R}$ | Real numbers |
| $\mathbb{R}_{\geq 0}$ | Non-negative real numbers |
| $\mathbb{R}_{>0}$ | Positive real numbers |
| $\mathbb{R}^3$ | Three-dimensional real vector space |
| $\mathbb{H}$ | Quaternion algebra |
| $\mathrm{SO}(3)$ | Set of right-handed three-dimensional rotation matrices |

[Table 3](#table-3) defines the semantic symbols used in the equations.

<a id="table-3"></a>

Table 3: Semantic symbols used in equations throughout this LX.

| Symbol | Meaning | Type | Units |
| --- | --- | --- | --- |
| $\mathrm{S}$, $\mathrm{B}$, $\mathrm{W}$ | Sensor, Duckiedrone body, and stationary world frame labels | Frame labels | - |
| $i$ | Cartesian component index | $i \in \{x, y, z\}$ | - |
| $\mathbf{R}_{\mathrm{WB}}$, $\mathbf{R}_{\mathrm{BS}}$, $\mathbf{R}_i(\theta)$ | Body-to-world rotation, sensor-to-body rotation, and right-handed elementary rotation about axis $i$ through angle $\theta$ | $\mathbf{R}_{\mathrm{WB}}, \mathbf{R}_{\mathrm{BS}}, \mathbf{R}_i(\theta) \in \mathrm{SO}(3)$ | - |
| $\mathbf{I}$ | Three-by-three identity matrix | Dimensionless matrix | - |
| $\mathbf{p}$, $p_i$ | Coordinates of a point and its component on the chosen axes | $\mathbf{p} \in \mathbb{R}^3$, $p_i \in \mathbb{R}$ | Chosen length units |
| $\mathbf{e}_i$ | Unit basis vector along Cartesian axis $i$ | $\mathbf{e}_i \in \mathbb{R}^3$, $\lVert\mathbf{e}_i\rVert = 1$ | - |
| $\mathbf{0}$ | Additive identity vector | $\mathbf{0} \in \mathbb{R}^3$ | Units of the surrounding vector |
| $\mathbf{a}$, $a_i$ | Linear acceleration and the component with index $i$ on the chosen axes | $\mathbf{a} \in \mathbb{R}^3$, $a_i \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $\mathbf{v}$, $v_i$ | Linear velocity and the component with index $i$ on the chosen axes | $\mathbf{v} \in \mathbb{R}^3$, $v_i \in \mathbb{R}$ | $\mathrm{m/s}$ |
| $\mathbf{x}$, $x_i$ | Position and the component with index $i$ on the chosen axes | $\mathbf{x} \in \mathbb{R}^3$, $x_i \in \mathbb{R}$ | $\mathrm{m}$ |
| $v_k$ | Dead-reckoned vertical velocity at sample $k$ | $v_k \in \mathbb{R}$ | $\mathrm{m/s}$ |
| $x_k$ | Dead-reckoned altitude at sample $k$ | $x_k \in \mathbb{R}$ | $\mathrm{m}$ |
| $\mathbf{f}$, $f_i$ | Proper acceleration and the component with index $i$ on the chosen axes | $\mathbf{f} \in \mathbb{R}^3$, $f_i \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $\tilde{\mathbf{f}}$, $\tilde{f}_i$ | Measured proper acceleration and the component with index $i$ on the chosen axes | $\tilde{\mathbf{f}} \in \mathbb{R}^3$, $\tilde{f}_i \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $\mathbf{b}_{a}$, $b_{a,i}$, $b$ | Accelerometer bias, its component, and the constant one-axis alias used in the drift example | $\mathbf{b}_{a} \in \mathbb{R}^3$, $b_{a,i}, b \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $\mathbf{n}_{a}$, $n_{a,i}$ | Random proper-acceleration measurement noise and the component with index $i$ on the chosen axes | $\mathbf{n}_{a} \in \mathbb{R}^3$, $n_{a,i} \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $\boldsymbol{\omega}$, $\omega_i$ | Angular velocity and the component with index $i$ on the chosen axes | $\boldsymbol{\omega} \in \mathbb{R}^3$, $\omega_i \in \mathbb{R}$ | $\mathrm{rad/s}$ |
| $\tilde{\boldsymbol{\omega}}$, $\tilde{\omega}_i$ | Measured angular velocity and the component with index $i$ on the chosen axes | $\tilde{\boldsymbol{\omega}} \in \mathbb{R}^3$, $\tilde{\omega}_i \in \mathbb{R}$ | $\mathrm{rad/s}$ |
| $\theta$, $\theta_i$ | Generic rotation angle and angle of rotation about axis $i$ | $\theta, \theta_i \in \mathbb{R}$ | $\mathrm{rad}$ |
| $\alpha$, $\beta$, $\gamma$ | Roll, pitch, and yaw in the ZYX Euler-angle convention | $\alpha, \beta, \gamma \in \mathbb{R}$ | $\mathrm{rad}$ |
| $\psi$ | Axis-angle rotation angle | $\psi \in \mathbb{R}$ | $\mathrm{rad}$ |
| $\mathbf{u}$, $u_x$, $u_y$, $u_z$ | Unit axis of an axis-angle rotation and its Cartesian components | $\mathbf{u} = (u_x, u_y, u_z)^{\mathsf{T}} \in \mathbb{R}^3$, $\lVert\mathbf{u}\rVert = 1$ | - |
| $q$, $\mathbf{q}_{\mathrm{v}}$, $q_x$, $q_y$, $q_z$, $q_w$ | Unit quaternion, its vector and scalar parts, and scalar-last ROS components | $q = (\mathbf{q}_{\mathrm{v}}, q_w) \in \mathbb{H}$, $\lVert q\rVert = 1$; $\mathbf{q}_{\mathrm{v}} \in \mathbb{R}^3$; $q = (q_x, q_y, q_z, q_w)$ | - |
| $\mathbf{b}_{\omega}$, $b_{\omega,i}$ | Gyroscope bias and the component with index $i$ on the chosen axes | $\mathbf{b}_{\omega} \in \mathbb{R}^3$, $b_{\omega,i} \in \mathbb{R}$ | $\mathrm{rad/s}$ |
| $\mathbf{n}_{\omega}$, $n_{\omega,i}$ | Random angular-velocity measurement noise and the component with index $i$ on the chosen axes | $\mathbf{n}_{\omega} \in \mathbb{R}^3$, $n_{\omega,i} \in \mathbb{R}$ | $\mathrm{rad/s}$ |
| $\mathbf{g}$, $g_i$ | Gravitational acceleration and the component with index $i$ on the chosen axes | $\mathbf{g} \in \mathbb{R}^3$, $g_i \in \mathbb{R}$ | $\mathrm{m/s^2}$ |
| $g_0$, $\mathrm{m}g_0$ | Standard-gravity magnitude and its milligravity unit | $g_0 \in \mathbb{R}_{>0}$ | $\mathrm{m/s^2}$ |
| $\varepsilon$ | One-axis position error due to a constant accelerometer bias | $\varepsilon \in \mathbb{R}$ | $\mathrm{m}$ |
| $t$ | Continuous time | $t \in \mathbb{R}_{\geq 0}$ | $\mathrm{s}$ |
| $s$, $\tau$ | Dummy continuous-time integration variables | $s, \tau \in \mathbb{R}_{\geq 0}$ | $\mathrm{s}$ |
| $k$ | Sample index | $k \in \mathbb{Z}_{\geq 0}$ | - |
| $j$ | Local post-calibration sample index | $j \in \mathbb{Z}_{\geq 0}$ | - |
| $t_k$ | Time of sample $k$ | $t_k \in \mathbb{R}_{\geq 0}$ | $\mathrm{s}$ |
| $\tau_j$ | Elapsed time from the start of integration to local sample $j$ | $\tau_j \in \mathbb{R}_{\geq 0}$ | $\mathrm{s}$ |
| $\Delta t_k$ | Elapsed time from sample $k - 1$ to sample $k$ | $\Delta t_k \in \mathbb{R}_{>0}$ | $\mathrm{s}$ |
| $\Delta t$ | Fixed sample interval, when $\Delta t_k = \Delta t$ | $\Delta t \in \mathbb{R}_{>0}$ | $\mathrm{s}$ |
| $N$ | Number of calibration samples | $N \in \mathbb{Z}_{>0}$ | - |
| $T_{\mathrm{cal}}$ | Calibration-window duration | $T_{\mathrm{cal}} \in \mathbb{R}_{>0}$ | $\mathrm{s}$ |
| $\hat{r}$ | Stationary reference for the measured vertical proper acceleration | $\hat{r} \in \mathbb{R}$ | $\mathrm{m/s^2}$ |

## What does an IMU measure?

[Figure 1](#figure-1) shows the two sensors packaged inside the IMU.

<figure id="figure-1" class="lx-figure" style="width:420px;">
  <img src="../assets/_images/imu.png" style="width:100%;" />
  <figcaption>
    Figure 1: The two sensors inside an IMU: the accelerometer reports a noisy proper-acceleration
      measurement along the sensor's three axes (straight arrows), while the gyroscope reports a noisy
      angular-velocity measurement about them (curved arrows). See the <a
      href="https://api.semanticscholar.org/CorpusID:6415932">image source</a>.
  </figcaption>
</figure>

[Table 4](#table-4) compares the quantities reported by those sensors.

<a id="table-4"></a>

Table 4: Measurements provided by a typical two-sensor IMU.

| Sensor | What it measures | Units |
| --- | --- | --- |
| Accelerometer | Noisy proper-acceleration measurement along sensor $x$, $y$, and $z$ axes | $\mathrm{m/s^2}$ |
| Gyroscope | Noisy angular-velocity measurement about sensor $x$, $y$, and $z$ axes | $\mathrm{rad/s}$ |

Some IMUs add a magnetometer, which measures the Earth's magnetic field and so provides an absolute heading. The Duckiedrone's IMU does not include one.

### The accelerometer

At time $t$, let $\mathbf{x}(t)$ be the sensor's position relative to the chosen stationary world frame. Its __linear velocity__ $\mathbf{v}(t) = \dot{\mathbf{x}}(t)$ is the rate of change of position, and its __linear acceleration__ is $\mathbf{a}(t) = \dot{\mathbf{v}}(t)$. Let $\mathbf{g}$ be gravitational acceleration. The __proper acceleration__ $\mathbf{f}(t)$ is:

<span id="equation-1"></span>

$$
\mathbf{f}(t) = \mathbf{a}(t) - \mathbf{g}.
\tag{1}
$$

This is a relation between vectors, not a declaration of a coordinate frame. It can be written in world, body, or sensor components, provided every term uses the same frame. In the sensor frame, write $\mathbf{f}_{\mathrm{S}}(t)$, $\mathbf{a}_{\mathrm{S}}(t)$, and $\mathbf{g}_{\mathrm{S}}(t)$ for the components of proper, linear, and gravitational acceleration:

<span id="equation-2"></span>

$$
\mathbf{f}_{\mathrm{S}}(t)
= \mathbf{a}_{\mathrm{S}}(t) - \mathbf{g}_{\mathrm{S}}(t).
\tag{2}
$$

An ideal accelerometer reports the proper acceleration $\mathbf{f}_{\mathrm{S}}(t)$ exactly. A real accelerometer reports the __measured proper acceleration__ $\tilde{\mathbf{f}}_{\mathrm{S}}(t)$, with a constant accelerometer bias $\mathbf{b}_{a,\mathrm{S}}$ and random proper-acceleration measurement noise $\mathbf{n}_{a,\mathrm{S}}(t)$:

<span id="equation-3"></span>

$$
\tilde{\mathbf{f}}_{\mathrm{S}}(t)
= \mathbf{f}_{\mathrm{S}}(t)
+ \mathbf{b}_{a,\mathrm{S}}
+ \mathbf{n}_{a,\mathrm{S}}(t).
\tag{3}
$$

[Equation (2)](#equation-2) gives the sensor-frame components. In [Equation (3)](#equation-3), the bias shifts the reading, while random noise makes it vary between samples.

An accelerometer realizes this measurement with a mass-spring-damper system: a small mass of known size on a spring, inside a box. Accelerate the box and the mass lags behind, stretching the spring. How far it stretches gives an instantaneous proper-acceleration measurement, provided the elastic constant of the spring is known.

For this level-sensor example, $f_z$ and $\tilde{f}_z$ denote the $z$ components of the ideal and measured proper acceleration in sensor axes.

[Figure 2](#figure-2) illustrates the mass-spring-damper model.

<a id="figure-2"></a>

<p class="lx-figure" style="width:640px;">
  <img src="../assets/_images/mass_spring_damper.png" style="width:100%;" />
</p>

Figure 2: An ideal accelerometer as a mass-spring-damper. The box is fixed to the Duckiedrone, and the mass inside can only move by stretching the spring, so how far the spring stretches is what the sensor reads. At rest on a table, the spring supports the mass against gravitational acceleration, and an ideal accelerometer reports $\tilde{f}_z = f_z = +9.81\,\mathrm{m/s^2}$ upward. In ideal free fall, with negligible contact forces, the box and the mass fall together, the spring relaxes, and an ideal accelerometer reports $\tilde{f}_z = f_z = 0$.

Now leave the box sitting still on a table. What should the spring do?

It stays stretched. The spring supports the mass against gravitational acceleration, so the sensor reports an upward proper acceleration of $9.81\,\mathrm{m/s^2}$ rather than zero. This is the accelerometer's response to being supported against gravitational acceleration, not a direct measurement of gravitational acceleration. Throw the box off the table and, while air resistance is negligible, the opposite happens: mass and box fall together, the spring relaxes, and the reading drops toward zero. Once air resistance becomes significant, it pushes on the box, the spring does not fully relax, and the reading is not zero.

Thus, the accelerometer does not report how fast the Duckiedrone is moving, nor its linear acceleration. It reports a noisy __proper-acceleration measurement__: an estimate of proper acceleration relative to free fall. Here, __free fall__ means the sensor is subject only to gravitational acceleration. It does not require a literal vacuum, but contact forces such as air resistance must be negligible. In an ordinary fall through air, air resistance is a contact force and the accelerometer can read nonzero.

Near the Earth's surface, gravitational acceleration points down, towards the center of the Earth, with magnitude $9.81\,\mathrm{m/s^2}$. For a level sensor with its $z$ axis pointing up, the constant sensor-frame gravitational-acceleration vector $\mathbf{g}_{\mathrm{S}}$ is:

$$
\mathbf{g}_{\mathrm{S}}
=
\begin{bmatrix}
0 \\
0 \\
-9.81
\end{bmatrix}
\,\mathrm{m/s^2}.
$$

For a level, stationary sensor, the constant sensor-frame proper-acceleration vector $\mathbf{f}_{\mathrm{S}}$ is:

$$
\mathbf{f}_{\mathrm{S}}
=
\begin{bmatrix}
0 \\
0 \\
+9.81
\end{bmatrix}
\,\mathrm{m/s^2}.
$$

In the component expressions in [Equation (2)](#equation-2) and [Equation (3)](#equation-3), every vector must be expressed in the same frame. Let $\mathbf{R}_{\mathrm{BS}}$ be the fixed rotation from sensor-frame components to body-frame components. Write $\mathbf{f}_{\mathrm{B}}(t)$ and $\tilde{\mathbf{f}}_{\mathrm{B}}(t)$ for the corresponding body-frame proper acceleration and measured proper acceleration, so $\mathbf{f}_{\mathrm{B}}(t) = \mathbf{R}_{\mathrm{BS}}\,\mathbf{f}_{\mathrm{S}}(t)$ and $\tilde{\mathbf{f}}_{\mathrm{B}}(t) = \mathbf{R}_{\mathrm{BS}}\,\tilde{\mathbf{f}}_{\mathrm{S}}(t)$. Let $\mathbf{I}$ be the three-by-three identity matrix. When the sensor and body axes align, $\mathbf{R}_{\mathrm{BS}} = \mathbf{I}$. Ignoring or correcting measurement error, recovering world-frame linear acceleration from the sensor-frame proper acceleration requires the fixed sensor-to-body rotation and the Duckiedrone's orientation. Let $\mathbf{R}_{\mathrm{WB}}(t)$ be the rotation from body-frame components to world-frame components; write $\mathbf{a}_{\mathrm{W}}(t)$ and $\mathbf{g}_{\mathrm{W}}$ for the world-frame linear and gravitational accelerations:

$$
\mathbf{a}_{\mathrm{W}}(t)
= \mathbf{R}_{\mathrm{WB}}(t)\,\mathbf{R}_{\mathrm{BS}}\,\mathbf{f}_{\mathrm{S}}(t)
+ \mathbf{g}_{\mathrm{W}}.
$$

Replacing proper acceleration with the measured proper acceleration gives:

<span id="equation-4"></span>

$$
\mathbf{a}_{\mathrm{W}}(t)
\approx
\mathbf{R}_{\mathrm{WB}}(t)\,\mathbf{R}_{\mathrm{BS}}\,\tilde{\mathbf{f}}_{\mathrm{S}}(t)
+ \mathbf{g}_{\mathrm{W}}.
\tag{4}
$$

[Equation (4)](#equation-4) first rotates measured sensor-frame proper acceleration into body-frame components with $\mathbf{R}_{\mathrm{BS}}$, then into world-frame components with $\mathbf{R}_{\mathrm{WB}}(t)$. [Notebook 3](./3-rotation-representations.ipynb) develops those rotations; the gravitational-acceleration term is then added back in the world frame.

After correcting the proper-acceleration measurement for sensor error, rotating it into the world frame, and adding gravitational acceleration, the result is an estimate of $\mathbf{a}_{\mathrm{W}}(t)$.

Let $\mathbf{x}_{\mathrm{W}}(0)$ and $\mathbf{v}_{\mathrm{W}}(0)$ be the initial world-frame position and linear velocity, and let $\mathbf{x}_{\mathrm{W}}(t)$ be the position at time $t$. Use $s$ and $\tau$ as dummy integration times. Integrating the world-frame linear acceleration twice gives:

<span id="equation-5"></span>

$$
\mathbf{x}_{\mathrm{W}}(t)
= \mathbf{x}_{\mathrm{W}}(0)
+ \mathbf{v}_{\mathrm{W}}(0)t
+ \int_0^t \int_0^\tau \mathbf{a}_{\mathrm{W}}(s)\,\mathrm{d}s\,\mathrm{d}\tau.
\tag{5}
$$

This double integration forms a dead-reckoning estimate, which accumulates sensor error. [Notebook 2](./2-imu-estimation-and-drift.ipynb) shows why bias makes its position estimate drift.

Let $\mathbf{0}$ denote the zero vector. The contrast is easiest to see with level sensor axes whose positive $z$ direction points up. At rest on a table, $\mathbf{a}(t) = \mathbf{0}$ but $\mathbf{f}(t) = -\mathbf{g}$. For an ideal accelerometer, $\tilde{\mathbf{f}}_{\mathrm{S}}(t) = \mathbf{f}_{\mathrm{S}}(t)$, so it reports $\tilde{f}_z = +9.81\,\mathrm{m/s^2}$. In ideal free fall, with negligible contact forces, $\mathbf{a}(t) = \mathbf{g}$ and $\mathbf{f}(t) = \mathbf{0}$, so the measured value is close to zero. Let $a_z$ be the upward component of linear acceleration. If the Duckiedrone accelerates upward at $a_z = +2\,\mathrm{m/s^2}$, the proper acceleration is $f_z = +11.81\,\mathrm{m/s^2}$ and $\tilde{f}_z$ should be close to it.

This distinction also explains a potentially misleading ROS field name. `linear_acceleration` in `sensor_msgs/Imu` identifies a translational-acceleration field rather than an angular one; its name alone does not say whether a gravitational-acceleration correction has been applied. In the [PX4](https://docs.px4.io/main/)/[MAVROS2](https://github.com/mavlink/mavros) stream used here, it contains a proper-acceleration measurement, not the world-frame acceleration $\mathbf{a}_{\mathrm{W}}$. The message's `frame_id` identifies the axes of its components.

Whether the sensor's axes line up with the Duckiedrone's own is a separate question; [Notebook 5](./5-duckiedrone-imu-architecture.ipynb#from-the-imu-chip-to-px4) shows where PX4 rotates the readings into the Duckiedrone's axes.

### The gyroscope

Consider a rotation constrained to one fixed sensor axis $i$. Let $\theta_i(t)$ be the signed angle of that rotation about axis $i$. The corresponding __angular velocity__ $\omega_i(t)$ is the rate of change of $\theta_i(t)$. For general three-dimensional motion, angular velocity is the vector $\boldsymbol{\omega}(t)$. An ideal gyroscope reports the angular velocity $\boldsymbol{\omega}_{\mathrm{S}}(t)$ in its sensor frame exactly. A real gyroscope reports the __measured angular velocity__ $\tilde{\boldsymbol{\omega}}_{\mathrm{S}}(t)$, with a constant gyroscope bias $\mathbf{b}_{\omega,\mathrm{S}}$ and random angular-velocity measurement noise $\mathbf{n}_{\omega,\mathrm{S}}(t)$:

<span id="equation-6"></span>

$$
\tilde{\boldsymbol{\omega}}_{\mathrm{S}}(t)
= \boldsymbol{\omega}_{\mathrm{S}}(t)
+ \mathbf{b}_{\omega,\mathrm{S}}
+ \mathbf{n}_{\omega,\mathrm{S}}(t).
\tag{6}
$$

In [Equation (6)](#equation-6), the bias shifts the reading and random noise makes it vary between samples.

For the one-axis rotation above:

$$
\omega_i(t)
= \dot{\theta}_i(t).
$$

Let $\theta_i(0)$ be the initial angle of rotation about axis $i$. Integrating the angular velocity $\omega_i(t)$ recovers the angle at time $t$:

$$
\theta_i(t)
= \theta_i(0) + \int_0^t \omega_i(\tau)\,\mathrm{d}\tau.
$$

In practice, the resulting orientation estimate also accumulates sensor error. A residual constant bias $b_{\omega,i}$ produces an angle error that grows as $b_{\omega,i}t$; random noise adds further error. For general three-dimensional motion, the roll, pitch, and yaw rates are coupled rather than equal to $\omega_x$, $\omega_y$, and $\omega_z$. Orientation must instead be integrated as a rotation, for example with the unit quaternion $q$ introduced in [Notebook 4](./4-quaternions-and-rotation-representations.ipynb).

Intuitively, imagine being spun around slowly in an office chair with closed eyes. How fast the chair is turning at any moment is relatively easy to feel. Saying which way it ended up facing after a while of spinning is not: that requires adding the rate up over the whole turn.

## Gathering IMU measurements from a Duckiedrone

Before going further into what the numbers mean, it is worth looking at them. The IMU data arrives on the ROS 2 topic `/mavros/imu/data`, and this section shows what one message on that topic looks like. [Notebook 7](./7-interfacing-with-imu.ipynb) shows how to read the topic on a Duckiedrone. A virtual Duckiedrone publishes the same message, with values produced by the simulator rather than by a sensor; the Duckiematrix has to be running, with the Duckiedrone attached to an entity, or the topic stays silent.

The message below was captured on a physical Duckiedrone resting motionless on a bench:

```yaml
header:
  stamp:
    sec: 1788950655
    nanosec: 961939565
  frame_id: base_link
orientation:
  x: 0.0035833670220291654
  y: 0.003122046427906428
  z: -0.7076347443857675
  w: -0.7065624650101054
orientation_covariance:
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
angular_velocity:
  x: 8.995339157991111e-06
  y: -0.0005893486086279153
  z: 0.000459553673863411
angular_velocity_covariance:
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
linear_acceleration:
  x: -0.018726181238889694
  y: -0.0861072316765773
  z: 9.91747760772705
linear_acceleration_covariance:
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
---
```
This message is of the type `sensor_msgs/Imu`, and it carries three groups of numbers. Its
  `frame_id: base_link` says that the components in this example use the Duckiedrone body frame,
  $\text{B}$.

Two of the groups come from the IMU. `linear_acceleration` holds the __measured proper acceleration__ $\tilde{\mathbf{f}}_{\mathrm{B}} = (\tilde{f}_x, \tilde{f}_y, \tilde{f}_z)^{\mathsf{T}}$, with every component measured in $\mathrm{m/s^2}$. It is a noisy proper-acceleration measurement of the proper acceleration $\mathbf{f}_{\mathrm{B}}$, not the world-frame acceleration $\mathbf{a}_{\mathrm{W}}$. `angular_velocity` holds the __measured angular velocity__ $\tilde{\boldsymbol{\omega}}_{\mathrm{B}} = (\tilde{\omega}_x, \tilde{\omega}_y, \tilde{\omega}_z)^{\mathsf{T}}$, with every component measured in $\mathrm{rad/s}$. It is a noisy angular-velocity measurement of the angular velocity. The third, `orientation`, is not a measurement at all: no sensor in the package reports an angle. It is computed by the flight controller, and [Notebook 2](./2-imu-estimation-and-drift.ipynb#why-the-imu-data-matters) explains how and why. Each group is followed by a covariance array, which describes how much the publisher believes its own numbers.

Now compare those readings against the stationary values. The Duckiedrone was still and level. The expected proper acceleration is:

$$
\mathbf{f}_{\mathrm{B}}
=
\begin{bmatrix}
0 \\
0 \\
+9.81
\end{bmatrix}
\,\mathrm{m/s^2}.
$$

The measured proper acceleration is approximately:

$$
\tilde{\mathbf{f}}_{\mathrm{B}}
\approx \mathbf{f}_{\mathrm{B}}.
$$

The measured angular velocity is approximately:

$$
\tilde{\boldsymbol{\omega}}_{\mathrm{B}}
\approx
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
\,\mathrm{rad/s}.
$$

The vertical reading is $9.917\,\mathrm{m/s^2}$, not $9.81\,\mathrm{m/s^2}$. The horizontal readings are $-0.019\,\mathrm{m/s^2}$ and $-0.086\,\mathrm{m/s^2}$, not zero. The angular rates are small but not zero either. Nothing was moving, yet a physical measurement is not exactly its expected stationary value.

That gap is not a mistake in the theory or a fault in the Duckiedrone. Every physical sensor adds an error to what it measures, and [Notebook 2](./2-imu-estimation-and-drift.ipynb) measures that error and shows what it does to an estimate built from the readings.

## Further reading

The [sensor_msgs/Imu message definition](https://github.com/ros2/common_interfaces/blob/jazzy/sensor_msgs/msg/Imu.msg) describes the measured fields and the flight-controller orientation estimate.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()